# Kaggle Submission: reference blend baseline

This notebook is intended for direct upload into the BirdCLEF+ 2026 Kaggle competition.

What you need before running it:
- Attach the competition data.
- Attach a Kaggle Dataset that contains `LB862.pt` and `LB872.pt`.
- Keep the notebook on CPU with internet disabled.

What it does:
- Finds the competition data and reference checkpoint dataset.
- Loads a reference EfficientNet-B0 + attention SED model.
- Runs 5-second soundscape inference.
- Applies the reference blend and optional public-LB heuristics.
- Saves `/kaggle/working/submission.csv`.

Important note:
- During interactive editing, hidden `test_soundscapes` may not be present yet.
- During `Save Version` and `Submit`, Kaggle mounts the real hidden test files and this notebook should then create the final submission.


In [ ]:
import os
import re
import time
import warnings
from dataclasses import dataclass
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import torchaudio.transforms as T

warnings.filterwarnings("ignore")
START = time.time()
torch.set_num_threads(max(1, os.cpu_count() or 1))
print("Torch:", torch.__version__)
print("CPU threads:", torch.get_num_threads())


## Configuration

- `EXP_ID` controls blend weights.
- `APPLY_REFERENCE_HEURISTICS` turns on temporal smoothing and file-max leakage.
- `MODEL_DATASET_HINT` is optional. Set it only if you know the exact dataset folder name under `/kaggle/input/`.


In [ ]:
# Blend options match the reference notebook.
EXP_ID = 2
APPLY_REFERENCE_HEURISTICS = True
MODEL_DATASET_HINT = None  # Example: "birdclef-2026-model"

BLEND_OPTIONS = [
    ("finetuned_only", {"mode": "prob", "ft": 1.0, "base": 0.0}),
    ("baseline_only", {"mode": "prob", "ft": 0.0, "base": 1.0}),
    ("prob_ft80_base20", {"mode": "prob", "ft": 0.8, "base": 0.2}),
    ("prob_ft70_base30", {"mode": "prob", "ft": 0.7, "base": 0.3}),
    ("prob_ft50_base50", {"mode": "prob", "ft": 0.5, "base": 0.5}),
    ("logit_ft80_base20", {"mode": "logit", "ft": 0.8, "base": 0.2}),
    ("logit_ft70_base20", {"mode": "logit", "ft": 0.7, "base": 0.3}),
    ("logit_ft50_base50", {"mode": "logit", "ft": 0.5, "base": 0.5}),
]

assert 0 <= EXP_ID < len(BLEND_OPTIONS)
SELECTED_NAME, SELECTED_SPEC = BLEND_OPTIONS[EXP_ID]
print("Blend:", SELECTED_NAME)
print("Heuristics:", APPLY_REFERENCE_HEURISTICS)


In [ ]:
@dataclass
class Config:
    sr: int = 32_000
    chunk_duration: float = 5.0
    n_mels: int = 224
    n_fft: int = 2048
    hop_length: int = 512
    fmin: int = 0
    fmax: int = 16_000
    top_db: float = 80.0
    power: float = 2.0
    norm: str = "slaney"
    mel_scale: str = "htk"
    backbone: str = "tf_efficientnet_b0.ns_jft_in1k"
    num_classes: int = 234
    in_channels: int = 3
    dropout: float = 0.1
    drop_path_rate: float = 0.0
    gem_p_init: float = 3.0

    @property
    def chunk_samples(self) -> int:
        return int(self.sr * self.chunk_duration)


def resolve_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026"),
    ]
    for candidate in candidates:
        if (candidate / "sample_submission.csv").exists():
            return candidate
    raise FileNotFoundError("BirdCLEF competition data was not found under /kaggle/input.")


def resolve_model_root(model_dataset_hint: str | None = None) -> Path:
    input_root = Path("/kaggle/input")
    required = ["LB862.pt", "LB872.pt"]

    def has_required_files(folder: Path) -> bool:
        return folder.is_dir() and all((folder / name).exists() for name in required)

    if model_dataset_hint:
        hinted = input_root / model_dataset_hint
        if has_required_files(hinted):
            return hinted

    for child in sorted(input_root.iterdir()):
        if has_required_files(child):
            return child

    available = sorted(path.name for path in input_root.iterdir())
    raise FileNotFoundError(
        "Could not find a Kaggle Dataset containing LB862.pt and LB872.pt. "
        f"Available inputs: {available}"
    )


cfg = Config()
DATA_ROOT = resolve_data_root()
MODEL_ROOT = resolve_model_root(MODEL_DATASET_HINT)
TEST_DIR = DATA_ROOT / "test_soundscapes"
TRAIN_SC_DIR = DATA_ROOT / "train_soundscapes"
BASELINE_CKPT = MODEL_ROOT / "LB862.pt"
FINETUNED_CKPT = MODEL_ROOT / "LB872.pt"
device = torch.device("cpu")

sample_sub = pd.read_csv(DATA_ROOT / "sample_submission.csv")
SPECIES = sample_sub.columns[1:].tolist()
cfg.num_classes = len(SPECIES)

print("DATA_ROOT:", DATA_ROOT)
print("MODEL_ROOT:", MODEL_ROOT)
print("Species:", len(SPECIES))
print("Baseline ckpt:", BASELINE_CKPT.name)
print("Finetuned ckpt:", FINETUNED_CKPT.name)


## Model Blocks

The architecture below matches the available reference checkpoints: EfficientNet-B0 backbone, GeM pooling across frequency, and an attention-style SED head.


In [ ]:
class GEMFreqPool(nn.Module):
    def __init__(self, p_init: float = 3.0, eps: float = 1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p_init))
        self.eps = eps

    def forward(self, x):
        p = self.p.clamp(min=1.0)
        x = x.clamp(min=self.eps).pow(p)
        x = x.mean(dim=2)
        return x.pow(1.0 / p)


class AttentionSEDHead(nn.Module):
    def __init__(self, feat_dim: int, num_classes: int, dropout: float = 0.1):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(feat_dim, feat_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        self.att_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)
        self.cls_conv = nn.Conv1d(feat_dim, num_classes, kernel_size=1)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.fc(x)
        x = x.permute(0, 2, 1)
        att = torch.tanh(self.att_conv(x))
        att = F.softmax(att, dim=-1)
        cls = self.cls_conv(x)
        clipwise_logit = (att * cls).sum(dim=-1)
        clipwise_prob = torch.sigmoid(clipwise_logit)
        return {
            "clipwise_logit": clipwise_logit,
            "clipwise_prob": clipwise_prob,
        }


class SEDModel(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.backbone = timm.create_model(
            cfg.backbone,
            pretrained=False,
            in_chans=cfg.in_channels,
            features_only=False,
            global_pool="",
            num_classes=0,
            drop_path_rate=cfg.drop_path_rate,
        )
        feat_dim = self.backbone.num_features
        self.gem_pool = GEMFreqPool(p_init=cfg.gem_p_init)
        self.head = AttentionSEDHead(feat_dim, cfg.num_classes, cfg.dropout)

    def forward(self, x):
        features = self.backbone(x)
        pooled = self.gem_pool(features)
        return self.head(pooled)


class MelSpectrogramTransform(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.mel = T.MelSpectrogram(
            sample_rate=cfg.sr,
            n_fft=cfg.n_fft,
            hop_length=cfg.hop_length,
            n_mels=cfg.n_mels,
            f_min=cfg.fmin,
            f_max=cfg.fmax,
            power=cfg.power,
            norm=cfg.norm,
            mel_scale=cfg.mel_scale,
        )
        self.db = T.AmplitudeToDB(stype="power", top_db=cfg.top_db)

    @torch.no_grad()
    def forward(self, waveforms):
        waveforms = torch.nan_to_num(waveforms.float(), nan=0.0, posinf=0.0, neginf=0.0)
        mel = self.mel(waveforms)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=0.0, neginf=0.0)
        mel = self.db(mel)
        mel = torch.nan_to_num(mel, nan=-80.0, posinf=0.0, neginf=-80.0)
        batch_size = mel.shape[0]
        mel_flat = mel.reshape(batch_size, -1)
        mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(-1)
        mel = (mel - mel_min) / (mel_max - mel_min + 1e-7)
        mel = torch.nan_to_num(mel, nan=0.0, posinf=1.0, neginf=0.0)
        return mel.unsqueeze(1).repeat(1, 3, 1, 1)


## Load Checkpoints

This cell reconstructs the reference model architecture and loads both checkpoints.


In [ ]:
def safe_load_checkpoint(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def extract_state_dict(ckpt):
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        return ckpt["model_state_dict"]
    return ckpt


def load_model(ckpt_path: Path, tag: str):
    ckpt = safe_load_checkpoint(ckpt_path)
    model = SEDModel(cfg)
    model.load_state_dict(extract_state_dict(ckpt), strict=True)
    model.to(device).eval()
    print(f"Loaded {tag}: {ckpt_path.name}")
    return model


baseline_model = load_model(BASELINE_CKPT, "baseline")
finetuned_model = load_model(FINETUNED_CKPT, "finetuned")
mel_transform = MelSpectrogramTransform(cfg).to(device).eval()


## Build Submission Row Mapping

The notebook uses the sample submission row IDs to reconstruct the expected output order. When hidden test files are not mounted yet, the notebook can still run and will fall back to an empty prediction map until Kaggle provides the real test soundscapes during scoring.


In [ ]:
row_pattern = re.compile(r"^(.*)_(\d+)$")


def parse_row_id(row_id: str):
    match = row_pattern.match(str(row_id))
    if not match:
        return None, None
    return match.group(1), int(match.group(2))


expected_ids = sample_sub["row_id"].tolist()
expected_by_stem = {}
for row_id in expected_ids:
    stem, end_sec = parse_row_id(row_id)
    if stem is None:
        continue
    expected_by_stem.setdefault(stem, []).append(end_sec)

for stem in expected_by_stem:
    expected_by_stem[stem] = sorted(expected_by_stem[stem])

test_files = sorted(TEST_DIR.glob("*.ogg"))

if len(test_files) == 0:
    print("No hidden test audio is currently mounted. This is normal before Kaggle scoring starts.")
    fallback_files = []
    for stem in sorted(expected_by_stem):
        candidate = TRAIN_SC_DIR / f"{stem}.ogg"
        if candidate.exists():
            fallback_files.append(candidate)
    test_files = fallback_files
    print(f"Aligned fallback files found: {len(test_files)}")
else:
    print(f"Hidden test files found: {len(test_files)}")


## Inference Helpers

The default blend is `0.8 * finetuned + 0.2 * baseline`. When `APPLY_REFERENCE_HEURISTICS` is enabled, the notebook also applies confidence-sharpened smoothing and file-level leakage from the reference public-LB notebook.


In [ ]:
def load_soundscape(path: Path) -> np.ndarray:
    audio, _ = librosa.load(path, sr=cfg.sr, mono=True)
    return np.nan_to_num(audio, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def probs_from_blend(base_logits, ft_logits, base_probs, ft_probs):
    if SELECTED_SPEC["mode"] == "prob":
        probs = SELECTED_SPEC["ft"] * ft_probs + SELECTED_SPEC["base"] * base_probs
    else:
        logits = SELECTED_SPEC["ft"] * ft_logits + SELECTED_SPEC["base"] * base_logits
        probs = torch.sigmoid(logits)
    probs = torch.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
    probs = torch.clamp(probs, 0.0, 1.0)
    return probs.detach().cpu().numpy()


def apply_heuristics(probs: np.ndarray) -> np.ndarray:
    if len(probs) > 4:
        sharpen_power = 1.5
        probs_sharp = probs ** sharpen_power
        weights = np.array([0.05, 0.15, 0.60, 0.15, 0.05], dtype=np.float32)
        padded = np.pad(probs_sharp, ((2, 2), (0, 0)), mode="edge")
        smoothed = (
            weights[0] * padded[:-4]
            + weights[1] * padded[1:-3]
            + weights[2] * padded[2:-2]
            + weights[3] * padded[3:-1]
            + weights[4] * padded[4:]
        )
        probs = smoothed ** (1.0 / sharpen_power)
    elif len(probs) > 2:
        weights = np.array([0.20, 0.60, 0.20], dtype=np.float32)
        padded = np.pad(probs, ((1, 1), (0, 0)), mode="edge")
        probs = (
            weights[0] * padded[:-2]
            + weights[1] * padded[1:-1]
            + weights[2] * padded[2:]
        )
    file_max = np.max(probs, axis=0, keepdims=True)
    return probs + 0.05 * file_max


CHUNK = cfg.chunk_samples
all_row_ids = []
all_preds = []


In [ ]:
print("Running inference...")

with torch.no_grad():
    for file_idx, path in enumerate(test_files, start=1):
        stem = path.stem
        audio = load_soundscape(path)

        if stem in expected_by_stem:
            n_chunks = max(expected_by_stem[stem]) // int(cfg.chunk_duration)
        else:
            n_chunks = max(1, len(audio) // CHUNK)

        padded_len = n_chunks * CHUNK
        if len(audio) < padded_len:
            audio = np.pad(audio, (0, padded_len - len(audio)))
        else:
            audio = audio[:padded_len]

        chunks = audio.reshape(n_chunks, CHUNK)
        chunk_tensor = torch.from_numpy(chunks).float().to(device)
        mel = mel_transform(chunk_tensor)

        base_out = baseline_model(mel)
        ft_out = finetuned_model(mel)

        base_probs = torch.nan_to_num(base_out["clipwise_prob"], nan=0.0, posinf=1.0, neginf=0.0)
        ft_probs = torch.nan_to_num(ft_out["clipwise_prob"], nan=0.0, posinf=1.0, neginf=0.0)

        probs = probs_from_blend(
            base_logits=base_out["clipwise_logit"],
            ft_logits=ft_out["clipwise_logit"],
            base_probs=base_probs,
            ft_probs=ft_probs,
        )

        if APPLY_REFERENCE_HEURISTICS:
            probs = apply_heuristics(probs)

        if stem in expected_by_stem:
            valid_indices = [
                (end_sec // int(cfg.chunk_duration)) - 1
                for end_sec in expected_by_stem[stem]
            ]
            valid_indices = [idx for idx in valid_indices if 0 <= idx < n_chunks]
            target_row_ids = [f"{stem}_{(idx + 1) * int(cfg.chunk_duration)}" for idx in valid_indices]
        else:
            valid_indices = list(range(n_chunks))
            target_row_ids = [f"{stem}_{(idx + 1) * int(cfg.chunk_duration)}" for idx in valid_indices]

        all_row_ids.extend(target_row_ids)
        all_preds.extend(probs[valid_indices])

        if file_idx % 25 == 0 or file_idx == len(test_files):
            elapsed = time.time() - START
            print(f"Processed {file_idx}/{len(test_files)} files | elapsed={elapsed:.1f}s")

print("Inference complete.")


## Build Submission

When hidden test files are mounted, this cell writes the final competition file to `/kaggle/working/submission.csv`.


In [ ]:
def build_submission(row_ids, preds, expected_ids, species):
    if len(preds) == 0:
        pred_df = pd.DataFrame(
            np.zeros((0, len(species)), dtype=np.float32),
            columns=species,
            index=pd.Index([], name="row_id"),
        )
    else:
        pred_df = pd.DataFrame(
            np.asarray(preds, dtype=np.float32),
            columns=species,
            index=pd.Index(row_ids, name="row_id"),
        )

    pred_df = pred_df[~pred_df.index.duplicated(keep="first")]

    missing_ids = [row_id for row_id in expected_ids if row_id not in pred_df.index]
    if missing_ids:
        zeros = pd.DataFrame(
            np.zeros((len(missing_ids), len(species)), dtype=np.float32),
            columns=species,
            index=pd.Index(missing_ids, name="row_id"),
        )
        pred_df = pd.concat([pred_df, zeros], axis=0)

    pred_df = pred_df.loc[expected_ids]
    return pred_df.reset_index()


submission = build_submission(all_row_ids, all_preds, expected_ids, SPECIES)
submission.to_csv("/kaggle/working/submission.csv", index=False)
submission.to_csv(f"/kaggle/working/submission_{SELECTED_NAME}.csv", index=False)

print("Saved: /kaggle/working/submission.csv")
print(f"Saved: /kaggle/working/submission_{SELECTED_NAME}.csv")
print("Shape:", submission.shape)
print("Total runtime (sec):", round(time.time() - START, 1))
submission.head()


## What To Do On Kaggle

1. Upload this notebook to the BirdCLEF+ 2026 competition.
2. Attach the competition data.
3. Attach your checkpoint dataset containing `LB862.pt` and `LB872.pt`.
4. Run `Save Version` and choose `Save & Submit`.
5. After Kaggle finishes scoring, copy the public LB score back into your project notes.
